# Fase 1: Calibración de Hiperparámetros mediante Optimización Bayesiana

## 1. Introducción y Marco Teórico

En el presente documento se aborda la calibración sistemática de los hiperparámetros que gobiernan el comportamiento del algoritmo de Optimización por Colonia de Hormigas (ACO) y la Búsqueda Tabú (TS), aplicados al Problema de Enrutamiento de Arcos Capacitados (CARP) para el vehículo autónomo "Hykue".

### Hipótesis de Calibración (H1)

*Existe una configuración óptima de hiperparámetros en el algoritmo ACO (pesos $\alpha$, $\beta$, tasa de evaporación $\rho$ y penalización $\Omega$) que equilibra de manera eficiente la explotación de la memoria colectiva y la exploración heurística, minimizando de forma consistente la función objetivo al mitigar los ciclos redundantes inducidos por callejones sin salida y penalizaciones de deadheading.*


### Función Objetivo Unificada

Para evitar sesgos de terminación temprana, el fitness se evalúa bajo la siguiente definición matemática inmutable:
$$Z = \frac{B_{max}}{\text{Arcos Únicos Inspeccionados}}$$
Donde un menor valor de $Z$ representa una mayor eficiencia de cobertura dentro de la restricción dura de la autonomía energética ($B_{max} = 3000.0$).

---

## 2. Metodología de Optimización con Optuna

Para explorar el espacio combinatorio de parámetros se utiliza **Optuna**, un framework de optimización de hiperparámetros. Específicamente, se configura el muestreador **Tree-structured Parzen Estimator (TPE)**.

**Justificación de TPE frente a Grid Search:**
A diferencia de una búsqueda exhaustiva en cuadrícula que sufre de explosión combinatoria, TPE aplica Optimización Bayesiana aplicando un enfoque probabilístico. Modela de forma independiente la distribución de las configuraciones que arrojan buenos resultados frente a las que arrojan malos resultados, guiando el muestreo hacia las regiones de mayor rendimiento del espacio de búsqueda continuo y discreto.

In [1]:
import os
import gc
import joblib
import numpy as np
import pandas as pd
import optuna
import optuna.visualization as vis

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [2]:
import sys
from pathlib import Path

src_path = Path().resolve().parent / "src"
sys.path.append(str(src_path))

from aco import ACO_CARP
from tabu import TabuSearch_CARP
from graph_model import RedTuberias

In [3]:
# Validación de existencia de datos
RUTA_RED_BASE = "../data/red_base_5x5.json"
if not os.path.exists(RUTA_RED_BASE):
    raise FileNotFoundError(f"Debe ejecutar 'graph_generator.py' primero para generar: {RUTA_RED_BASE}")

print("Entorno inicializado correctamente.")

Entorno inicializado correctamente.


---

## 3. Calibración del Algoritmo Principal: ACO

Definiremos la función objetivo para Optuna. Para mitigar el ruido estocástico inherente a las metaheurísticas poblacionales, cada evaluación de un "trial" ejecutará el algoritmo bajo **3 semillas pseudoaleatorias distintas** (42, 123, 999), retornando la **mediana empírica** del fitness final como métrica de robustez.

Los rangos de exploración han sido acotados bajo criterios teóricos:
- $\alpha$ (Intensificación de feromona): $[0.5, 3.0]$
- $\beta$ (Visibilidad heurística local): $[1.0, 5.0]$
- $\rho$ (Tasa de evaporación global): $[0.05, 0.50]$
- $\Omega$ (Factor de penalización): $[10.0, 2000.0]$

In [4]:
RUTA_ESTUDIO_ACO = "../data/calibracion_aco.pkl"

def objetivo_aco(trial):
    
    alpha = trial.suggest_float("alpha", 0.5, 3.0)
    beta = trial.suggest_float("beta", 1.0, 5.0)
    rho = trial.suggest_float("rho", 0.05, 0.5)
    omega = trial.suggest_float("omega", 10.0, 5000.0)
    
    # Parámetros estructurales fijos para la fase de calibración
    N_HORMIGAS = 20
    MAX_ITER = 200
    BATERIA_MAX = 3000.0
    
    semillas = [19, 42, 123]
    fitness_ejecuciones = []
    
    for semilla in semillas:
        np.random.seed(semilla)

        # Carga del grafo
        red = RedTuberias.cargar_red(RUTA_RED_BASE)
        
        aco = ACO_CARP(
            red=red,
            n_hormigas=N_HORMIGAS,
            max_iter=MAX_ITER,
            alpha=alpha,
            beta=beta,
            rho=rho,
            Q=1.0,
            omega=omega,
            bateria_max=BATERIA_MAX
        )
        
        resultado = aco.run()
        fitness_ejecuciones.append(resultado.fitness_final)
        
        # Gestión explícita de memoria en Jupyter
        del red, aco
        gc.collect()
        
    return np.median(fitness_ejecuciones)

In [5]:
# Ejecución del estudio de calibración ACO
if os.path.exists(RUTA_ESTUDIO_ACO):
    print("Estudio de ACO ya existe en memoria. Continuando con el análisis...")
    estudio_aco = joblib.load(RUTA_ESTUDIO_ACO)
else:
    print("Ejecutando Búsqueda de Hiperparámetros para ACO")
    print()
    estudio_aco = optuna.create_study(study_name="calibracion_aco", direction="minimize")
    estudio_aco.optimize(objetivo_aco, n_trials=50)

Ejecutando Búsqueda de Hiperparámetros para ACO

--- Iniciando ACO ---
Población: 20 hormigas | Iteraciones: 200
Batería Máxima: 3000.0 | Factor de Castigo: 2285.7837135173195

Iteración 1 -> Nuevo Óptimo | Z: 176.47 | Cobertura: 17 tramos | Batería consumida: 3000.00
Detención temprana en iteración 66 por falta de mejora.

--- Resultados Finales ---
Mejor Puntaje Z (Costo por tramo): 176.47
Tramos únicos inspeccionados: 17 / 40
Batería consumida total: 3000.00 / 3000.0
Batería consumida en deadheading: 300.00
Ruta propuesta:
[0, 1, 2, 3, 4, 9, 8, 7, 6, 1, 0, 5, 10, 15, 20, 21, 16, 11, 6, 7, 8]
--- Iniciando ACO ---
Población: 20 hormigas | Iteraciones: 200
Batería Máxima: 3000.0 | Factor de Castigo: 2285.7837135173195

Iteración 1 -> Nuevo Óptimo | Z: 176.47 | Cobertura: 17 tramos | Batería consumida: 3000.00
Detención temprana en iteración 66 por falta de mejora.

--- Resultados Finales ---
Mejor Puntaje Z (Costo por tramo): 176.47
Tramos únicos inspeccionados: 17 / 40
Batería consum

In [6]:
print("\nACO:")
print(f"- Mejores Parámetros: {estudio_aco.best_params}")
print(f"- Mediana del Fitness del Mejor Estudio: {estudio_aco.best_value:.4f}")


ACO:
- Mejores Parámetros: {'alpha': 0.6713782862174057, 'beta': 2.0158052630221213, 'rho': 0.11590493823268709, 'omega': 2933.4222250362777}
- Mediana del Fitness del Mejor Estudio: 166.6667


In [7]:
# coordenadas paralelas para visualizar el espacio de búsqueda
fig_optuna = vis.plot_parallel_coordinate(
    estudio_aco, 
    params=["alpha", "beta", "rho", "omega"]
)

fig_optuna.update_layout(
    title="Análisis del Espacio de Búsqueda para ACO",
)

fig_optuna.show()

In [8]:
# Ejecución de ACO con los mejores parámetros encontrados

red = RedTuberias.cargar_red(RUTA_RED_BASE)
aco = ACO_CARP(
    red=red,
    n_hormigas=20,
    max_iter=200,
    alpha=estudio_aco.best_params["alpha"],
    beta=estudio_aco.best_params["beta"],
    rho=estudio_aco.best_params["rho"],
    Q=1.0,
    omega=estudio_aco.best_params["omega"],
    bateria_max=3000
)

resultado = aco.run()

del red, aco

--- Iniciando ACO ---
Población: 20 hormigas | Iteraciones: 200
Batería Máxima: 3000 | Factor de Castigo: 2933.4222250362777

Iteración 1 -> Nuevo Óptimo | Z: 176.47 | Cobertura: 17 tramos | Batería consumida: 2940.00
Iteración 42 -> Nuevo Óptimo | Z: 166.67 | Cobertura: 18 tramos | Batería consumida: 3000.00
Detención temprana en iteración 66 por falta de mejora.

--- Resultados Finales ---
Mejor Puntaje Z (Costo por tramo): 166.67
Tramos únicos inspeccionados: 18 / 40
Batería consumida total: 3000.00 / 3000
Batería consumida en deadheading: 0.00
Ruta propuesta:
[0, 1, 6, 11, 16, 21, 20, 15, 10, 5, 6, 7, 2, 3, 4, 9, 8, 7, 12]


---

## 4. Calibración de la Línea Base: Tabu Search

Para asegurar una comparación simétrica y justa en la Fase 2, se aplica el mismo protocolo bayesiano sobre los componentes lógicos de Tabu Search:
- `t_vecindad` (Tamaño de la muestra aleatoria de vecinos): Se explora discretamente en el rango $[5, 50]$.
- `tenencia_tabu` (Duración de la persistencia en la lista de exclusión): Rango entero $[2, 30]$.
- $\Omega$ (Factor de penalización homólogo para balancear la exploración): Rango continuo $[10.0, 2000.0]$.

In [9]:
RUTA_ESTUDIO_TABU = "../data/calibracion_tabu.pkl"

def objetivo_tabu(trial):
    
    t_vecindad = trial.suggest_int("t_vecindad", 5, 50)
    tenencia_tabu = trial.suggest_int("tenencia_tabu", 2, 30)
    omega = trial.suggest_float("omega", 10.0, 2000.0)
    
    MAX_ITER = 1000  # TS al ser de trayectoria única requiere más iteraciones base
    BATERIA_MAX = 3000.0
    
    semillas = [19, 42, 123]
    fitness_ejecuciones = []
    
    for semilla in semillas:
        np.random.seed(semilla)
        red = RedTuberias.cargar_red(RUTA_RED_BASE)
        
        ts = TabuSearch_CARP(
            red_tuberias=red,
            bateria_maxima=BATERIA_MAX,
            max_iter=MAX_ITER,
            t_vecindad=t_vecindad,
            tenencia_tabu=tenencia_tabu,
            omega=omega
        )
        
        resultado = ts.run()
        fitness_ejecuciones.append(resultado.fitness_final)
        
        del red, ts
        gc.collect()
        
    return np.median(fitness_ejecuciones)

In [10]:
# Ejecución del estudio de calibración TS
if os.path.exists(RUTA_ESTUDIO_TABU):
    print("Estudio de Tabu Search ya existe en memoria. Continuando con el análisis...")
    estudio_tabu = joblib.load(RUTA_ESTUDIO_TABU)
else:
    print("Ejecutando Búsqueda de Hiperparámetros para Tabu Search")
    print()
    estudio_tabu = optuna.create_study(study_name="calibracion_tabu", direction="minimize")
    estudio_tabu.optimize(objetivo_tabu, n_trials=50)

Ejecutando Búsqueda de Hiperparámetros para Tabu Search

--- Iniciando Tabu Search ---
Iteraciones: 1000 | Tenencia Tabú: 18
Batería Máxima: 3000.0 | Factor de Castigo: 972.2698477757965

Iteración 1 -> Nuevo Óptimo | Z: 200.00 | Cobertura: 15 tramos | Batería consumida: 2980.00
Iteración 22 -> Nuevo Óptimo | Z: 187.50 | Cobertura: 16 tramos | Batería consumida: 2920.00

Convergencia temprana en iteración 122.

--- Resultados Finales ---
Mejor Puntaje Z (Costo por tramo): 187.50
Tramos únicos inspeccionados: 16 / 40
Batería consumida total: 2920.00
Batería en deadheading: 100.00
Tiempo de cómputo: 0.2173 seg
Ruta propuesta:
[0, 5, 10, 11, 6, 1, 2, 3, 8, 7, 6, 5, 10, 15, 20, 21, 16, 11]
--- Iniciando Tabu Search ---
Iteraciones: 1000 | Tenencia Tabú: 18
Batería Máxima: 3000.0 | Factor de Castigo: 972.2698477757965

Iteración 1 -> Nuevo Óptimo | Z: 200.00 | Cobertura: 15 tramos | Batería consumida: 2960.00
Iteración 3 -> Nuevo Óptimo | Z: 176.47 | Cobertura: 17 tramos | Batería consumida

In [11]:
print("\nTABÚ SEARCH:")
print(f"- Mejores Parámetros: {estudio_tabu.best_params}")
print(f"- Mediana del Fitness del Mejor Estudio: {estudio_tabu.best_value:.4f}")


TABÚ SEARCH:
- Mejores Parámetros: {'t_vecindad': 30, 'tenencia_tabu': 8, 'omega': 256.3559667986333}
- Mediana del Fitness del Mejor Estudio: 176.4706


In [12]:
# coordenadas paralelas para visualizar el espacio de búsqueda
fig_optuna = vis.plot_parallel_coordinate(
    estudio_tabu, 
    params=["t_vecindad", "tenencia_tabu", "omega"]
)

fig_optuna.update_layout(
    title="Análisis del Espacio de Búsqueda para Tabu Search",
)

fig_optuna.show()

In [13]:
# Ejecución de Tabú Search con los mejores parámetros encontrados

red = RedTuberias.cargar_red(RUTA_RED_BASE)
tabu = TabuSearch_CARP(
    red_tuberias=red,
    bateria_maxima=3000.0,
    max_iter=1000,
    t_vecindad=estudio_tabu.best_params["t_vecindad"],
    tenencia_tabu=estudio_tabu.best_params["tenencia_tabu"],
    omega=estudio_tabu.best_params["omega"]
)

resultado = tabu.run()

del red, tabu

--- Iniciando Tabu Search ---
Iteraciones: 1000 | Tenencia Tabú: 8
Batería Máxima: 3000.0 | Factor de Castigo: 256.3559667986333

Iteración 8 -> Nuevo Óptimo | Z: 200.00 | Cobertura: 15 tramos | Batería consumida: 3000.00

Convergencia temprana en iteración 108.

--- Resultados Finales ---
Mejor Puntaje Z (Costo por tramo): 200.00
Tramos únicos inspeccionados: 15 / 40
Batería consumida total: 3000.00
Batería en deadheading: 300.00
Tiempo de cómputo: 0.4851 seg
Ruta propuesta:
[0, 1, 6, 11, 12, 7, 6, 5, 10, 15, 16, 21, 16, 11, 6, 1, 2, 3, 4]


---

## 5. Conclusiones de la Fase 1 y Exportación de Parámetros


Los parámetros obtenidos representan los vectores de configuración óptimos bajo las condiciones topológicas de nuestra red de prueba base. Comprobando así que la hipótesis de calibración (H1) propuesta más arriba es verdadera, ya que logramos identificar un conjunto de hiperparámetros que optimizan el rendimiento de ambos algoritmos bajo el uso de la misma función objetivo.

Estos valores serán inyectados de forma estática en el script de la **Fase 2** para iniciar la recolección masiva de datos (15 ejecuciones por instancia).

Procedemos a exportar los estudios completos en formato `.pkl` para su posterior análisis o inclusión de tablas de sensibilidad en el informe final.

In [14]:
# guardamos los estudios para volverlos a cargar sin necesidad de recalibrar cada vez

joblib.dump(estudio_aco, RUTA_ESTUDIO_ACO)
joblib.dump(estudio_tabu, RUTA_ESTUDIO_TABU)

print("Estudios exportados exitosamente a la carpeta ../data/")

Estudios exportados exitosamente a la carpeta ../data/
